Импорты и пути

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

ROOT = Path("/home/jupyter/project")
#сохраненная модель и список признаков,которые использовались при ее обучении
MODEL_PATH = ROOT / "model" / "final_model.joblib"
FEATURES_PATH = ROOT / "model" / "feature_names.json"
#входная таблица для модели
INPUT_CSV = ROOT / "data" / "inference_features.csv"
#папка и файл с предсказаниями
OUTPUT_DIR = ROOT / "output"
OUTPUT_CSV = OUTPUT_DIR / "predictions.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#порог, после которого считаем прогноз ПТСР положительным
THRESHOLD = 0.5
BOOTSTRAP_ITERATIONS = 2000
#фиксируем random state для воспроизводимости
RANDOM_STATE = 42

print("ROOT:", ROOT)
print("MODEL:", MODEL_PATH)
print("FEATURES:", FEATURES_PATH)
print("INPUT:", INPUT_CSV)
print("OUTPUT:", OUTPUT_CSV)

ROOT: /home/jupyter/project
MODEL: /home/jupyter/project/model/final_model.joblib
FEATURES: /home/jupyter/project/model/feature_names.json
INPUT: /home/jupyter/project/data/inference_features.csv
OUTPUT: /home/jupyter/project/output/predictions.csv


In [2]:
SOURCE_CSV = ROOT / "eeg_subject_level_features.csv"

print("Looking for source file:")
print(SOURCE_CSV)
#проверяем, что исходная таблица существует
if not SOURCE_CSV.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_CSV}")
#загружаем точный список признаков, на которых обучалась финальная модель
with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)
#загружаем признаки всех участников
df_source = pd.read_csv(SOURCE_CSV)

print("Source shape:", df_source.shape)
print("Model expects:", len(FEATURE_NAMES), "features")
#проверяем наличие идентификатора участника
if "subject_id" not in df_source.columns:
    raise ValueError("Column 'subject_id' not found.")
#проверяем, что присутствуют все признаки,которые ожидает сохраненная модель
missing_features = [
    feature
    for feature in FEATURE_NAMES
    if feature not in df_source.columns
]
#оставляем только id и признаки в том же порядке,в котором они подавались модели при обучении
if missing_features:
    raise ValueError(
        f"Missing {len(missing_features)} model features:\n"
        + "\n".join(missing_features)
    )


inference_df = df_source[
    ["subject_id"] + FEATURE_NAMES
].copy()


INPUT_CSV.parent.mkdir(
    parents=True,
    exist_ok=True
)

inference_df.to_csv(
    INPUT_CSV,
    index=False
)

print("\nInference file created successfully:")
print(INPUT_CSV)

print("Shape:", inference_df.shape)
print("Number of features:", len(FEATURE_NAMES))

display(inference_df.head())

Looking for source file:
/home/jupyter/project/eeg_subject_level_features.csv
Source shape: (216, 810)
Model expects: 252 features

Inference file created successfully:
/home/jupyter/project/data/inference_features.csv
Shape: (216, 253)
Number of features: 252


,subject_id,Rest_O1_delta_log,Rest_O1_delta_rel,Rest_O1_theta_log,Rest_O1_theta_rel,Rest_O1_alpha_log,Rest_O1_alpha_rel,Rest_O1_beta_log,Rest_O1_beta_rel,Rest_T3_delta_log,Rest_T3_delta_rel,Rest_T3_theta_log,Rest_T3_theta_rel,Rest_T3_alpha_log,Rest_T3_alpha_rel,Rest_T3_beta_log,Rest_T3_beta_rel,Rest_Fp1_delta_log,Rest_Fp1_delta_rel,Rest_Fp1_theta_log,Rest_Fp1_theta_rel,Rest_Fp1_alpha_log,Rest_Fp1_alpha_rel,Rest_Fp1_beta_log,Rest_Fp1_beta_rel,Rest_Fp2_delta_log,Rest_Fp2_delta_rel,Rest_Fp2_theta_log,Rest_Fp2_theta_rel,Rest_Fp2_alpha_log,Rest_Fp2_alpha_rel,Rest_Fp2_beta_log,Rest_Fp2_beta_rel,Rest_T4_delta_log,Rest_T4_delta_rel,Rest_T4_theta_log,Rest_T4_theta_rel,Rest_T4_alpha_log,Rest_T4_alpha_rel,Rest_T4_beta_log,...,S1_Rest_O2_alpha_log,S1_Rest_O2_alpha_rel,S1_Rest_O2_beta_log,S1_Rest_O2_beta_rel,S1_Rest_Fp1_Fp2_delta_coherence,S1_Rest_Fp1_Fp2_theta_coherence,S1_Rest_Fp1_Fp2_alpha_coherence,S1_Rest_Fp1_Fp2_beta_coherence,S1_Rest_T3_T4_delta_coherence,S1_Rest_T3_T4_theta_coherence,S1_Rest_T3_T4_alpha_coherence,S1_Rest_T3_T4_beta_coherence,S1_Rest_O1_O2_delta_coherence,S1_Rest_O1_O2_theta_coherence,S1_Rest_O1_O2_alpha_coherence,S1_Rest_O1_O2_beta_coherence,S1_Rest_O1_aperiodic_exponent,S1_Rest_O1_aperiodic_offset,S1_Rest_T3_aperiodic_exponent,S1_Rest_T3_aperiodic_offset,S1_Rest_Fp1_aperiodic_exponent,S1_Rest_Fp1_aperiodic_offset,S1_Rest_Fp2_aperiodic_exponent,S1_Rest_Fp2_aperiodic_offset,S1_Rest_T4_aperiodic_exponent,S1_Rest_T4_aperiodic_offset,S1_Rest_O2_aperiodic_exponent,S1_Rest_O2_aperiodic_offset,S1_Rest_Fp1_Fp2_delta_asym,S1_Rest_T3_T4_delta_asym,S1_Rest_O1_O2_delta_asym,S1_Rest_Fp1_Fp2_theta_asym,S1_Rest_T3_T4_theta_asym,S1_Rest_O1_O2_theta_asym,S1_Rest_Fp1_Fp2_alpha_asym,S1_Rest_T3_T4_alpha_asym,S1_Rest_O1_O2_alpha_asym,S1_Rest_Fp1_Fp2_beta_asym,S1_Rest_T3_T4_beta_asym,S1_Rest_O1_O2_beta_asym
0,A003_68,-20.438562,0.555582,-21.778437,0.193787,-22.612213,0.087752,-22.603924,0.076001,-20.325837,0.587298,-21.824045,0.188002,-22.376060,0.076321,-22.723282,0.066672,-20.373255,0.560608,-21.816684,0.185061,-22.403740,0.084176,-22.635013,0.072327,-20.321035,0.571286,-21.766921,0.193750,-22.429728,0.085280,-22.641276,0.067576,-20.325889,0.587295,-21.824022,0.188013,-22.375967,0.076318,-22.723242,...,-0.005933,0.044415,0.428535,0.093834,0.002473,0.005975,0.004926,-0.048606,-1.627401e-08,-9.798154e-09,-1.707231e-08,-7.509658e-08,-0.038268,0.005062,0.002225,-0.020881,-0.383481,-0.658253,-0.322289,-0.620891,-0.371897,-0.665261,-0.332671,-0.633304,-0.322265,-0.620872,-0.322289,-0.620891,-0.050194,0.000035,-0.027160,0.021347,0.000070,-0.163596,-0.047317,0.000060,0.320293,0.027312,0.000054,-0.031599
1,A036_69,-24.430757,0.227124,-24.607220,0.193988,-24.946044,0.133067,-24.193923,0.294906,-24.441134,0.126151,-24.928305,0.084210,-25.217611,0.057538,-23.163321,0.440347,-23.457624,0.351637,-24.264189,0.188275,-24.859159,0.109270,-24.026839,0.204517,-23.459039,0.376576,-23.944920,0.241055,-24.537259,0.114578,-24.221373,0.158554,-24.001329,0.243140,-24.485418,0.142492,-24.847134,0.099420,-23.683118,...,-0.628719,-0.040740,0.269221,0.144506,0.272286,0.246618,0.234819,0.073419,8.859730e-01,8.437627e-01,4.487327e-01,3.259333e-02,0.667281,0.584048,0.374230,0.008658,1.174434,1.835191,0.348931,0.553266,1.046644,1.516764,0.830081,1.208372,1.003651,1.619044,0.934903,1.453288,-0.107072,1.228935,-0.297101,0.253361,0.378692,0.290765,0.383227,0.485577,0.355788,-0.103193,0.077680,0.043922
2,A037_69,-24.178981,0.221283,-23.986138,0.234941,-24.589189,0.129012,-23.813016,0.277955,-23.886940,0.331464,-24.595291,0.133225,-25.011195,0.075352,-23.912310,0.262719,-24.925288,0.167204,-25.060470,0.160608,-25.621246,0.075296,-24.043550,0.351361,-25.084475,0.171451,-24.908981,0.232613,-25.480340,0.107669,-24.449576,0.316948,-25.113156,0.170471,-24.898076,0.216198,-25.633185,0.096766,-24.405129,...,0.973959,0.002609,0.535410,-0.147547,-0.156154,-0.139971,-0.086007,-0.001218,-2.324611e-02,7.844724e-02,1.341814e-01,-3.872001e-02,0.163123,0.156757,0.180930,0.132856,0.30715

Для применения модели была сформирована таблица из 216 участников. Сохранённая модель ожидает 252 признака, и все необходимые признаки присутствуют в исходных данных. Итоговый входной файл содержит 253 столбца: subject_id и 252 признака модели. Это подтверждает соответствие структуры входных данных структуре, использованной моделью

In [3]:
#загружаем уже обученную финальную модель
model = joblib.load(MODEL_PATH)

print("Model loaded successfully:")
print(MODEL_PATH)

Model loaded successfully:
/home/jupyter/project/model/final_model.joblib


Сохранённая модель загружается из директории model/. На этапе инференса модель уже не обучается заново, а только применяется к подготовленным данным.

In [4]:
#загружаем подготовленные для применения данные
df = pd.read_csv(INPUT_CSV)
#оставляем признаки строго в том порядке,который использовался при обучении модели
X = df[FEATURE_NAMES].copy()
#приводим все признаки к числовому типу
#некорректные значения превращаются в NaN, которые дальше обрабатываются внутри сохраненного pipeline
for col in FEATURE_NAMES:
    X[col] = pd.to_numeric(X[col], errors="coerce")
#получаем вероятность принадлежности к классу ПТСР
ptsd_probability = model.predict_proba(X)[:, 1]
#формируем выходной файл
predictions = pd.DataFrame({
    "subject_id": df["subject_id"],
    "ptsd_probability": ptsd_probability
})
#сохраняем результат применения модели
predictions.to_csv(
    OUTPUT_CSV,
    index=False
)
print("Predictions saved to:")
print(OUTPUT_CSV)

display(predictions.head(10))

Predictions saved to:
/home/jupyter/project/output/predictions.csv


,subject_id,ptsd_probability
0,A003_68,1.000000
1,A036_69,0.000037
2,A037_69,0.999922
3,A038_67,0.999677
4,А004_68,0.999285
5,А005_67,0.997914
6,А006_67,0.987363
7,А007_68,0.999796
8,А008_70,0.823342
9,А009_70,0.810362


В результате получаем файл с 2 колонками: subject_id, ptsd_probability. При этом, несколько здоровых участников 65–70 лет получают очень высокую вероятность ПТСР. Например, у A003_68, A037_69, A038_67 и ряда других вероятность близка к 1. Это потенциальный недостаток специфичности разработанной модели. 

Объединяем предсказания с исходными группами

In [5]:
SOURCE_CSV = ROOT / "eeg_subject_level_features.csv"
#загружаем исходные данные с известными группами
df_source = pd.read_csv(SOURCE_CSV)
#загружаем полученные моделью вероятности
df_pred = pd.read_csv(OUTPUT_CSV)
#соединяем истинные группы и прогнозы по subject_id, эта таблица нужна нам для расчета метрик
df_eval = df_source.merge(
    df_pred,
    on="subject_id",
    how="inner"
)

print("размер датасета:", df_eval.shape)

print("\nгруппы:")
print(df_eval["group"].value_counts())

print("\nвероятность:")
print(
    df_eval["ptsd_probability"].min(),
    
    df_eval["ptsd_probability"].max()
)

размер датасета: (216, 811)

группы:
group
Норма            128
Соматоформные     63
ПТСР              25
Name: count, dtype: int64

вероятность:
3.6628025118515065e-05 0.9999999999996408


После объединения прогнозов с исходными метками получена выборка из 216 участников: 128 человек группы нормы, 63 участника с соматоформными расстройствами и 25 с ПТСР. Вероятности ПТСР изменяются практически от 0 до 1, поэтому модель выдаёт полноценную вероятностную оценку, а не только бинарный класс.

In [6]:
#основная диагностическая выборка: ПТСР против нормы

#для ROC-AUC и balanced accuracy используем только ПТСР и контрольную группу
df_diagnostic = df_eval[
    df_eval["group"].isin(["Норма", "ПТСР"])
].copy()

#ПТСР = 1, норма = 0
df_diagnostic["target"] = (
    df_diagnostic["group"] == "ПТСР"
).astype(int)


print("Total:", len(df_diagnostic))
print("PTSD:", int(df_diagnostic["target"].sum()))
print("Control:", int((df_diagnostic["target"] == 0).sum()))

Total: 153
PTSD: 25
Control: 128


ROC-AUC + bootstrap 95% CI, 2000 итераций

In [7]:
def bootstrap_auc_ci(
    y_true,
    y_prob,
    n_iterations=2000,
    random_state=42
):
    #переводим данные в numpy-массивы
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
#фиксируем генератор случайных чисел
    rng = np.random.default_rng(random_state)

    auc_values = []
#2000 раз формируем bootstrap-выборку
    for _ in range(n_iterations):

        indices = rng.integers(
            0,
            len(y_true),
            size=len(y_true)
        )

        y_boot = y_true[indices]
        p_boot = y_prob[indices]

        #ROC-AUC можно считать только если в выборке присутствуют оба класса
        if len(np.unique(y_boot)) < 2:
            continue

        auc_values.append(
            roc_auc_score(
                y_boot,
                p_boot
            )
        )

    auc_values = np.asarray(auc_values)
#ROC-AUC на исходной выборке
    auc = roc_auc_score(
        y_true,
        y_prob
    )
#нижняя и верхняя границы 95% доверительного интервала
    ci_low = np.percentile(
        auc_values,
        2.5
    )
    ci_high = np.percentile(
        auc_values,
        97.5
    )
    return auc, ci_low, ci_high

#берем только диагностическую выборку ПТСР/контроль
y_true = df_diagnostic["target"].to_numpy()
y_prob = df_diagnostic["ptsd_probability"].to_numpy()
#считаем ROC-AUC и bootstrap-ДИ
auc, auc_low, auc_high = bootstrap_auc_ci(
    y_true,
    y_prob,
    n_iterations=2000,
    random_state=42
)

print("ROC-AUC:", round(auc, 4))
print(
    "95% CI:",
    f"[{auc_low:.4f}, {auc_high:.4f}]"
)


ROC-AUC: 0.86
95% CI: [0.7980, 0.9156]


На выборке ПТСР/контроль модель показала ROC-AUC 0.860. По результатам 2000 bootstrap-итераций 95% доверительный интервал составил [0.798; 0.916]. Таким образом, модель демонстрирует способность разделять участников с ПТСР и норму по рассчитанной вероятности ПТСР. При этом достаточно широкий доверительный интервал отражает неопределённость оценки, что скорее всего также связано с ограниченным размером выборки

Balanced accuracy + confusion matrix + specificity

In [8]:
THRESHOLD = 0.5
#при вероятности >= 0.5 считаем прогноз положительным на ПТСР
y_pred = (
    y_prob >= THRESHOLD
).astype(int)
#сбалансированная точность учитывает качество классификации обоих классов
balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)
#строим матрицу ошибок
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()
#доля правильно распознанных контрольных участников
specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)
#доля контрольных участников, ошибочно отнесенных моделью к ПТСР
false_positive_rate = (
    fp / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)
#доля правильно обнаруженных случаев ПТСР
sensitivity = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else np.nan
)

print(f"Threshold:           {THRESHOLD}")
print(f"ROC-AUC:             {auc:.4f}")
print(f"ROC-AUC 95% CI:      [{auc_low:.4f}, {auc_high:.4f}]")
print(f"Balanced accuracy:   {balanced_acc:.4f}")
print(f"Sensitivity:         {sensitivity:.4f}")
print(f"Specificity:         {specificity:.4f}")
print(f"False positive rate: {false_positive_rate:.4f}")

print("\nConfusion matrix:")
print(cm)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Threshold:           0.5
ROC-AUC:             0.8600
ROC-AUC 95% CI:      [0.7980, 0.9156]
Balanced accuracy:   0.8125
Sensitivity:         1.0000
Specificity:         0.6250
False positive rate: 0.3750

Confusion matrix:
[[80 48]
 [ 0 25]]

TN: 80
FP: 48
FN: 0
TP: 25


При пороге классификации 0.5 сбалансированная точность составила 0.813. Модель правильно классифицировала все 25 случаев ПТСР (чувствительность 100%), однако правильно распознала как контроль 80 из 128 участников группы нормы (специфичность 62.5%). Ещё 48 здоровых участников получили ложноположительный прогноз ПТСР, что соответствует FPR 37.5%. Таким образом, модель демонстрирует высокую чувствительность, однако её ограничением является сравнительно большое количество ложноположительных результатов.

In [9]:
control_groups = [
    "Норма",
    "Соматоформные"
]

group_results = []

for group_name in control_groups:

    subset = df_eval[
        df_eval["group"] == group_name
    ].copy()

    if len(subset) == 0:
        continue

    probabilities = subset[
        "ptsd_probability"
    ].to_numpy()

    predictions_group = (
        probabilities >= THRESHOLD
    ).astype(int)

    n = len(subset)

    false_positives = int(
        predictions_group.sum()
    )

    true_negatives = n - false_positives

    group_specificity = (
        true_negatives / n
    )

    group_fpr = (
        false_positives / n
    )

    group_results.append({
        "group": group_name,
        "n": n,
        "false_positive_ptsd": false_positives,
        "true_negative": true_negatives,
        "specificity": group_specificity,
        "false_positive_rate": group_fpr
    })

group_metrics = pd.DataFrame(
    group_results
)
display(group_metrics)

,group,n,false_positive_ptsd,true_negative,specificity,false_positive_rate
0,Норма,128,48,80,0.62500,0.37500
1,Соматоформные,63,43,20,0.31746,0.68254


In [10]:
AGE_COLUMNS = [
    "age",
    "Age",
    "AGE",
    "возраст",
    "Возраст"
]

age_column = next(
    (
        col
        for col in AGE_COLUMNS
        if col in df_eval.columns
    ),
    None
)

if age_column is not None:

    df_eval["age_value"] = pd.to_numeric(
        df_eval[age_column],
        errors="coerce"
    )

    print(
        "Возраст получен из столбца",
        age_column
    )

else:
    df_eval["age_value"] = pd.to_numeric(
        df_eval["subject_id"]
        .astype(str)
        .str.extract(r"_(\d+)$")[0],
        errors="coerce"
    )

    print(
        "Возраст определен на основе subject_id suffix."
    )
print(
    "Участников с известным возрастом:",
    df_eval["age_value"].notna().sum()
)

print(
    "Участников без известного возраста:",
    df_eval["age_value"].isna().sum()
)


Возраст определен на основе subject_id suffix.
Участников с известным возрастом: 128
Участников без известного возраста: 88


In [11]:
#разделяем участников с известным возрастом на возрастные подгруппы
def make_age_group(age):

    if pd.isna(age):
        return np.nan

    if age < 40:
        return "<40"

    if age < 50:
        return "40-49"

    if age < 65:
        return "50-64"

    return "65+"


df_eval["age_group"] = (
    df_eval["age_value"]
    .apply(make_age_group)
)


print(
    df_eval["age_group"]
    .value_counts(dropna=False)
)

age_group
NaN      88
<40      85
65+      21
40-49    12
50-64    10
Name: count, dtype: int64


In [12]:
# ============================================================
#специфичность модели по возрастным подгруппам нормы

#для анализа возраста оставляем только группу нормы,
#потому что для нее известен индивидуальный возраст
normal_age = df_eval[
    (df_eval["group"] == "Норма")
    &
    (df_eval["age_group"].notna())
].copy()


age_results = []

for age_group, subset in normal_age.groupby(
    "age_group"
):

    probabilities = subset[
        "ptsd_probability"
    ].to_numpy()

    #получаем бинарный прогноз модели
    predictions_age = (
        probabilities >= THRESHOLD
    ).astype(int)

    n = len(subset)

    #так как это здоровая контрольная группа,
    #каждый прогноз ПТСР является ложноположительным
    false_positives = int(
        predictions_age.sum()
    )

    true_negatives = (
        n - false_positives
    )

    #доля правильно распознанных здоровых участников
    specificity = (
        true_negatives / n
        if n > 0
        else np.nan
    )

    #доля здоровых участников,
    #ошибочно классифицированных как ПТСР
    false_positive_rate = (
        false_positives / n
        if n > 0
        else np.nan
    )

    age_results.append({
        "age_group": age_group,
        "n": n,
        "true_negative": true_negatives,
        "false_positive_ptsd": false_positives,
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "mean_ptsd_probability": probabilities.mean(),
        "median_ptsd_probability": np.median(probabilities)
    })


age_metrics = pd.DataFrame(
    age_results
)


#задаем понятный порядок возрастных групп
age_order = [
    "<40",
    "40-49",
    "50-64",
    "65+"
]

age_metrics["age_group"] = pd.Categorical(
    age_metrics["age_group"],
    categories=age_order,
    ordered=True
)

age_metrics = (
    age_metrics
    .sort_values("age_group")
    .reset_index(drop=True)
)


display(age_metrics)

,age_group,n,true_negative,false_positive_ptsd,specificity,false_positive_rate,mean_ptsd_probability,median_ptsd_probability
0,<40,85,59,26,0.694118,0.305882,0.320330,0.039393
1,40-49,12,10,2,0.833333,0.166667,0.135047,0.028143
2,50-64,10,10,0,1.000000,0.000000,0.026941,0.015382
3,65+,21,1,20,0.047619,0.952381,0.908456,0.999623


Для здоровых младше 40 лет специфичность составляет 69.4%: 59 из 85 правильно распознаны как не-ПТСР, а 26 получили ложноположительный результат. Средняя вероятность ПТСР 0.320, медианная всего 0.039. Разница между средней и медианой говорит о том, что большинство имеют низкие вероятности, но есть некоторое количество участников с очень высокими значениями, которые тянут среднее вверх. Для 40–49 лет специфичность выше и составляет 83.3%, что соответствует только 2 ложноположительным результатам из 12. Для 50–64 лет на этой небольшой подгруппе модель правильно классифицировала всех 10 человек: specificity = 100%, FPR = 0%. Однако слишком малое кол-во людей в данной выборке не позволяет рассматривать это как доказательство идеального определния диагноза для всех людей данного возраста. Для группы старше 65 из 21 здорового участника 20 модель классифицирует как ПТСР. Specificity падает до 4.8%, FPR возрастает до 95.2%. При этом,  средняя PTSD probability = 0.908, а медианная = 0.9996. То есть для большинства этих участников модель не просто слегка пересекает порог 0.5, а выдаёт чрезвычайно высокую вероятность ПТСР.

In [13]:
AGE_METRICS_CSV = OUTPUT_DIR / "metrics_by_age.csv"

age_metrics.to_csv(
    AGE_METRICS_CSV,
    index=False,
    encoding="utf-8-sig"
)

#общие диагностические метрики считаем только для ПТСР и нормы
y_all = df_diagnostic["target"].to_numpy()
p_all = df_diagnostic["ptsd_probability"].to_numpy()
pred_all = (p_all >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_all,
    pred_all,
    labels=[0, 1]
).ravel()

overall_metrics = pd.DataFrame([{
    "n": len(df_diagnostic),
    "ptsd_n": int(y_all.sum()),
    "control_n": int((y_all == 0).sum()),
    "threshold": THRESHOLD,
    "roc_auc": roc_auc_score(y_all, p_all),
    "balanced_accuracy": balanced_accuracy_score(y_all, pred_all),
    "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
    "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
    "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else np.nan,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp
}])

OVERALL_METRICS_CSV = OUTPUT_DIR / "overall_metrics.csv"

overall_metrics.to_csv(
    OVERALL_METRICS_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Общие метрики сохранены:")
print(OVERALL_METRICS_CSV)



group_results = []

for group in ["Норма", "Соматоформные"]:

    subset = df_eval[df_eval["group"] == group]

    #y_group = subset["target"].to_numpy()
    p_group = subset["ptsd_probability"].to_numpy()
    pred_group = (p_group >= THRESHOLD).astype(int)
    

    control_n = len(subset)
    fp = int(pred_group.sum())
    tn = control_n - fp

    group_results.append({
        "group": group,
        "n": control_n,
        "threshold": THRESHOLD,
        "specificity": tn / control_n if control_n > 0 else np.nan,
        "false_positive_rate": fp / control_n if control_n > 0 else np.nan,
        "false_positive_ptsd_n": fp,
        "true_negative_n": tn
    })

specificity_by_group = pd.DataFrame(group_results)

GROUP_METRICS_CSV = OUTPUT_DIR / "specificity_by_group.csv"

specificity_by_group.to_csv(
    GROUP_METRICS_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Специфичность по группам сохранена:")
print(GROUP_METRICS_CSV)


FINAL_SUMMARY_CSV = OUTPUT_DIR / "final_metrics_summary.csv"

with open(FINAL_SUMMARY_CSV, "w", encoding="utf-8") as f:

    f.write("OVERALL METRICS\n")
    overall_metrics.to_csv(f, index=False)

    f.write("\n\nSPECIFICITY BY GROUP\n")
    specificity_by_group.to_csv(f, index=False)

    f.write("\n\nMETRICS BY AGE\n")
    age_metrics.to_csv(f, index=False)

print("\nВсе метрики сохранены:")
print(" -", OVERALL_METRICS_CSV)
print(" -", GROUP_METRICS_CSV)
print(" -", AGE_METRICS_CSV)
print(" -", FINAL_SUMMARY_CSV)

Общие метрики сохранены:
/home/jupyter/project/output/overall_metrics.csv
Специфичность по группам сохранена:
/home/jupyter/project/output/specificity_by_group.csv

Все метрики сохранены:
 - /home/jupyter/project/output/overall_metrics.csv
 - /home/jupyter/project/output/specificity_by_group.csv
 - /home/jupyter/project/output/metrics_by_age.csv
 - /home/jupyter/project/output/final_metrics_summary.csv


**Выводы по 3 этапу работы**

В рамках третьего этапа была загружена сохранённая модель и рассчитана вероятность ПТСР для каждого участника. Результаты сохранены в требуемом формате subject_id, ptsd_probability.
Модель показала хорошую способность разделять ПТСР и основную контрольную группу, однако выявлена недостаточная специфичность: часть здоровых участников ошибочно классифицируется как ПТСР. Особенно выражена эта проблема среди участников старшего возраста и в группе соматоформных расстройств.
Возрастной анализ также показал заметные различия между подгруппами, наиболее выраженные для группы 65+. Поэтому на следующем этапе необходимо проверить влияние возраста и других возможных конфаундеров на прогноз модели и оценить, насколько выявленные ЭЭГ-признаки специфичны именно для ПТСР.